# Partitioned scatterplot diagnostics

A pooled scatterplot can mix several relationships. Two protocols can produce slopes with opposite signs, leaving the combined correlation close to zero; a bimodal weekday cycle can blur into a weekend mound; or diagonal bands, curved arcs, and tight clusters can disappear behind points from another regime. Partitioned scatterplot diagnostics search for a replayable rule on observed context columns that makes the same scatterplot more structured on held-out rows.

Use `onceover.profile(df, detectors=["partition_scagnostics"])` to run the search integrated with the frame model and return `partitioned_shape` findings. For explicit pairs, context columns, or metrics, call the search directly:

```python
result = onceover.partition_scagnostics(
    df, pairs=[("x", "y")], partition_by=["protocol"], metrics=["monotonic"],
)
```

Both entry points run the same Rust search with content-keyed sampling and train/holdout splitting. Each finding returns a replayable rule, labeled pooled and within-group scores, row counts, and search coverage. `report.rows(id, df)` extracts the annotated partition rows (or the standout subgroup rows for single-subgroup findings). See the [runnable usage example](../usage/find_partitioned_shapes.ipynb) and [case studies](../case_studies/partition_scagnostics_cases.ipynb).

## What the search optimizes

For a pair of numeric measures, take the rows with finite values on both axes. A rule assigns every one of those rows to a group. The search evaluates two objectives:

- **Whole-partition gain (`partition_gain`)**: the size-weighted average of the group scores minus the score on the pooled rows, minus a complexity penalty of `0.02` for each group beyond two. Every group needs at least 24 rows and 10% of its half's complete pairs, and raw gain must reach `0.08` in each half.
- **Exceptional single subgroup (`standout_subgroup`)**: for binary rules, detects a single sharp subgroup down to 5% of rows (at least 24 rows for `monotonic`/`functional`, 32 for geometric metrics) in either branch when its score reaches `0.45` (`0.55` geometric) with lift at least `0.35` (`0.40` geometric) over the pooled cloud — or, for `monotonic`, a minority sign reversal (`|ρ_group| ≥ 0.60` with opposite sign to `|ρ_pooled| ≥ 0.20`).

| Score | Computation | Read it as |
| --- | --- | --- |
| `monotonic` | Squared Spearman correlation with exact group midranks (requiring ≥ 4 distinct values per axis per group) | How closely the two measurements follow a consistent increasing or decreasing order. Read signed ρ to distinguish directions. |
| `functional` | Bias-adjusted nonparametric rank-space $R^2$ ($\max(\eta^2_{\text{adj}}(Y \mid \text{bin}(X)),\, \eta^2_{\text{adj}}(X \mid \text{bin}(Y)))$) over 12 discovery-fitted quantile bins | How tightly one coordinate behaves as a (possibly non-monotonic or multi-peak) function of the other, invariant to monotone rescaling. |
| `binned_striated` | Fraction of occupied 12 × 12 centroid MST vertices with degree 2 and incident-edge cosine below −0.75 | How often local tree segments continue straight through a point (parallel bands or straight tracks). |
| `binned_stringy` | Weighted diameter divided by total edge length of the occupied 12 × 12 centroid MST | How strongly the cloud forms a thin, unbranched 1D curve (arcs, parabolas, loops). |
| `binned_clumpy` | Maximum MST cut gap $1 - \max_{e' \in R(e)} w(e') / w(e)$ via $O(V)$ rerooting DP, requiring the smaller side to hold ≥ max(10, 10%) of points | How strongly the cloud separates into distinct compact clusters without letting isolated outlier points spoof clumping. |

All three MST-based geometric scores (`binned_striated`, `binned_stringy`, `binned_clumpy`) share a single Prim minimum spanning tree per candidate on a robust **`[q01, q99]`-scaled 12 × 12 grid** (falling back to `[min, max]` when quantiles collapse), preventing single extreme outliers from compressing the cloud into edge cells. A **2D-span flat-collapse guard** requires at least 2 occupied bins on both axes (at least 3 for `striated` and `stringy`), so a subgroup where one axis becomes constant or nearly constant cannot score as a 1D curve or stripe. [Scagnostics Distributions](https://www.cs.uic.edu/~wilkinson/Publications/scagdist.pdf)

In [1]:
from IPython.display import display

# A balanced split with two steep monotone relationships of opposite sign.
pooled = 0.04
groups = [(120, 0.97), (120, -0.95)]
weighted = sum(n * rho**2 for n, rho in groups) / sum(n for n, _ in groups)
gain = f"{weighted - pooled**2:.3f}"
description = (
    f"For example, pooled ρ of +{pooled:.2f} and two equally sized groups "
    f"with ρ of +{groups[0][1]:.2f} and −{abs(groups[1][1]):.2f} give a gain of "
)
display({
    "text/html": f"<p>{description}<strong>{gain}</strong> on the squared scale.</p>",
    "text/markdown": f"{description}**{gain}** on the squared scale.",
}, raw=True)

For example, pooled ρ of +0.04 and two equally sized groups with ρ of +0.97 and −0.95 give a gain of 0.920 on the squared scale.

## Where the rules come from

- **Low-cardinality categoricals (2–4 observed levels)**: supply both full level partitions and one-versus-remainder splits (first observed level versus remainder for binary columns).
- **Higher-cardinality categoricals (5–32 observed levels)**: supply up to 12 frequency-ranked one-versus-remainder splits for levels meeting the discovery support floor, so columns such as `site`, `region`, `month`, or `territory_id` are searched rather than skipped wholesale.
- **Ordered numeric descriptors**: supply threshold cutoffs at their discovery 20th, 35th, 50th, 65th, and 80th percentiles.
- **Two-column crosses**: two binary categorical descriptors supply a four-way crossing (`2a + b`), capturing interactions even when neither descriptor alone improves the pooled score.

The standalone API uses string, Boolean, Categorical, and Enum columns for categorical rules and numeric columns for ordered cutoffs. Inside `profile(df, detectors=["partition_scagnostics"])`, the pass integrates with the frame model: it excludes `identifier`, `row_number`, `timestamp`, `timestamp_part`, `text`, and exact-identity duplicates from both axes and descriptors, promotes integer-coded `flag` and `dimension` columns (2–12 distinct values) to exact `levels` rules, requires ≥ 8 distinct values for auto-inferred plotted axes, and prioritizes target-involving and continuous pairs ahead of alphabetical order.

A rule's remainder catches unmatched or missing categorical values; missing numeric context goes to the greater-than branch. The two plotted columns are excluded as descriptors for their own pair. Review descriptor provenance before acting: a renamed proxy or a label derived from the two axes can still manufacture apparent structure.

## What holdout validation establishes

Row sampling and the 50/50 discovery–holdout split are **content-keyed** (`RowKeys` over canonical column ranks), so reordering rows or columns does not redraw the split on tables with repeated values. Discovery selects at most one winning rule per pair and metric. On the untouched holdout half, the runner replays the exact rule, verifies support and gain, and applies two statistical guards:

1. **All-metric range-restriction guard**: if the partitioning column strongly separates one axis (`η² > 0.35` on discovery) while a single pooled 10-bin smooth rank curve already explains ≥ 60% of holdout rank variance and per-group curves add < 6% extra $R^2$, the proposal is filtered out (`coverage.validation.range_restriction_filtered`). This suppresses slicing a single nonlinear curve (such as $y = x^2$ split by a noisy proxy of $x$) into monotone pieces while keeping genuine regime changes such as Iris or opposite slopes.
2. **Bonferroni-adjusted permutation test**: shuffles group labels across holdout rows while keeping each `(x, y)` point intact and preserving exact group sizes, using a column-name-hashed RNG seed per pair. The permutation statistic matches the objective (`weighted` gain for `partition_gain`, branch score or opposite-sign $\rho$ for `standout_subgroup`), with early stopping once the best possible adjusted p-value exceeds `0.05`. [Athey and Imbens on sample splitting](https://pmc.ncbi.nlm.nih.gov/articles/PMC4941430/) · [Phipson and Smyth on random permutation tests](https://arxiv.org/abs/1603.05766)

After validation, redundant echoes on the same pair are folded away (`functional`, `binned_stringy`, and tight `binned_striated` when `monotonic` already explains the pair; `binned_stringy` under strong `binned_clumpy`). Read surviving findings as exploratory statements about the joint distribution: repeated entities or serially dependent observations (such as consecutive hourly readings) still require unit-aware analysis before causal or inferential claims.

A latent mixture without observed context remains a separate problem. Grouping two independent uniform coordinates on [−1, 1] by the sign of their product creates within-group Spearman correlations of +0.75 and −0.75 by construction, including on fresh rows. Latent regimes require comparing held-out mixture likelihoods against flexible single-population models. [Tarpey and Petkova](https://pmc.ncbi.nlm.nih.gov/articles/PMC2897159/)

## Search cost and evidence

Caps scale with the call and budget: standalone `partition_scagnostics()` defaults to 4,096 rows, 16 pairs, 96 descriptors, and 128 rules; `profile(..., detectors=["partition_scagnostics"])` uses 32 pairs / 160 rules at `standard` and 8,192 rows / 64 pairs / 128 descriptors / 256 rules at `deep`, polling `deadline_seconds` cooperatively inside both discovery and permutation loops. Candidate memberships are prepared once across pairs. Rust presorts each axis once per half for linear-time exact group midranks (`monotonic`), evaluates `functional` in a single linear stack-allocated pass, and computes all three MST metrics (`binned_striated`, `binned_stringy`, `binned_clumpy`) from one Prim pass over at most 144 occupied centroids per group. Pairs run in parallel via Rayon.

In the release benchmark (`cargo run --release -p onceover-mines --example partition_scagnostics -- --benchmark`), presorting sped up 512 repeated four-group Spearman evaluations by **8.2×** (`71.3 ms → 8.6 ms`) with zero numeric discrepancy. Across 12 synthetic scenarios, recall reached **112 / 112 (100%)** across all seven planted signal families (opposite slopes, numeric cutoff, two-column cross, diagonal stripes, curved arc, four-cluster regimes, and a 10% subgroup inside a 10-level categorical), **0 / 32** on the sliced-parabola range-restriction control, and **0 / 288** false positives for `functional` across all null and range-restriction controls.

In practice, read the evidence in this order:

1. Identify the plotted columns, metric, objective (`partition_gain` vs. `standout_subgroup`), and exact rule.
2. Compare the pooled and labeled per-group held-out scores and row counts (`report.rows(id, df)` extracts the exact annotated rows).
3. Inspect caps, shortlisted candidates, `range_restriction_filtered`, and holdout outcomes in coverage.
4. Check the scatterplot and descriptor provenance before acting on the finding.

## Why the detector runs on request

An initial promotion trial on the 20-dataset gate benchmark kept shuffled false positives unchanged, but exposed layout sensitivity (position-based sampling and alphabetical pair caps increased layout churn from 130 to 154) and surfaced calendar encodings and row counters in short digests. Integrating the v2 engine with content-keyed sampling, the frame model, identity deduplication, and the flat-collapse and range-restriction guards eliminated those failure modes (0 column-order diffs across all 20 gated tables, 0 row-shuffle diffs across all 19 tables with repeated values, and calendar/row-counter noise removed). The detector remains opt-in (`detectors=["partition_scagnostics"]` or `onceover.partition_scagnostics(...)`) so default profiles stay compact while analysts and agents can invoke the full five-metric search whenever hidden 2D structure is worth checking. The [case studies](../case_studies/partition_scagnostics_cases.ipynb) show the updated behavior across real and controlled tables.